# Scenarios

This notebook illustrates the scenarios of the simulator and how they are created (DESIGN-SCENARIO.md).

A scenario is an **environment exp/run** (`data/expruns/environment/`). It determines:
- **the geometry:** the farm type (`typename`) and its `planting`;
- **the field models:** the parameters of TYLCV, CCR and soil moisture, or pictures to use instead;
- **the precomputation:** the seeds, the number of days to precompute, and whether the environment is cached.

The notebook works on a **temporary copy** of the environment exp/runs, with a temporary results path, so nothing is written into the repository or the data directory. Everything that evolves uses Miniberry farms. The full Waterberry farm is shown only as a geometry.

In [ ]:
import pathlib
import shutil
import sys
import tempfile
sys.path.append("..")

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import yaml
from matplotlib.colors import ListedColormap

from exp_run_config import Config
from water_berry_farm import MiniberryFarm, WaterberryFarm
from wbf_helper import create_wbfe, create_wbf, precompute_environment, uses_cache
Config.PROJECTNAME = "WaterBerryFarms"

workspace = pathlib.Path(tempfile.mkdtemp())
expruns = workspace / "expruns"
shutil.copytree(pathlib.Path(Config().get_exprun_path(), "environment"), expruns / "environment")
(workspace / "results").mkdir()
Config().set_exprun_path(expruns)
Config().set_results_path(workspace / "results")
scenarios = expruns / "environment"

def show_exprun(run):
    print(f"--- {run}.yaml\n" + (scenarios / f"{run}.yaml").read_text())

def show_fields(wbfe, days, fields=("tylcv", "ccr", "soil"), title=""):
    """Advances the environment and shows the fields at the given days (1.0 white: healthy / wet)"""
    fig, axes = plt.subplots(len(fields), len(days), figsize=(2.2 * len(days), 2.2 * len(fields)), squeeze=False)
    for j, day in enumerate(days):
        wbfe.proceed(day - wbfe.time)
        for i, field in enumerate(fields):
            axes[i, j].imshow(getattr(wbfe, field).value.T, origin="lower", cmap="gray", vmin=0, vmax=1)
            axes[i, j].set_title(f"{field} day {day}", fontsize=9)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 1. Geometries and plantings

There are four geometry types. A geometry is a list of patches, each with a crop or land type.
- **The Miniberry farms** are owned entirely by the farmer: strawberries in the bottom half, tomatoes in the top half.
- **The full Waterberry farm** is 6000 × 5000 cells. It has a diagonal strawberry/tomato border, a pond and a wetland, and is surrounded by four neighbors' fields, through which disease can spread into the farm.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, typename in zip(axes, ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]):
    farm = create_wbf({"typename": typename})
    farm.visualize(ax)
    ax.set_title(f"{typename}: {farm.width} x {farm.height}")
plt.show()

The `planting` key of the exp/run changes the crops of the patches before the farm is rasterized into a **type map**, which gives the crop of every cell:
- `standard` keeps the crops;
- `tomato-only` turns every strawberry patch into tomato;
- `strawberry-only` does the reverse.

The pond, the wetland and the owner's area are unchanged.

In [ ]:
plantings = ["standard", "tomato-only", "strawberry-only"]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, planting in zip(axes, plantings):
    farm = MiniberryFarm(scale=3)
    farm.replant(planting)
    farm.create_type_map()
    ax.imshow(farm.type_map.T, origin="lower", cmap=ListedColormap(["lightblue", "lightcoral"]), vmin=1, vmax=2)
    ax.set_title(f"Miniberry-30, {planting}")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, planting in zip(axes, ["standard", "tomato-only"]):
    farm = WaterberryFarm()
    farm.replant(planting)
    for patch in farm.patches:  # color the patches by their (new) crop
        patch["polygon"].set_color({"strawberry": "lightblue", "tomato": "lightcoral"}.get(patch["type"], patch["color"]))
    farm.visualize(ax)
    ax.add_patch(plt.Rectangle((1000, 1000), 4000, 3000, fill=False, edgecolor="black", linewidth=2))
    ax.set_title(f"Waterberry, {planting} (black: the owner's area)")
plt.show()

## 2. Model-generated scenarios

A scenario's exp/run lists only what differs from `_defaults_environment.yaml`. Here are the defaults, and two Miniberry-30 scenarios that differ only in their planting:

In [ ]:
print((scenarios / "_defaults_environment.yaml").read_text())
show_exprun("miniberry-30")
show_exprun("miniberry-30-tomato")

`precompute_environment(run)` evolves a cached environment for its `precompute-time` days, unless this was already done with the same configuration. `create_wbfe(exp_env)` then gives the farm and the environment, which replays the precomputed days.

In the standard scenario, TYLCV spreads on the tomato half and CCR on the strawberry half. In the tomato-only scenario, TYLCV can spread over the whole farm, and there is no CCR.

In [ ]:
for run in ["miniberry-30", "miniberry-30-tomato"]:
    farm, wbfe = create_wbfe(precompute_environment(run))
    show_fields(wbfe, [1, 4, 7, 10], title=run)

## 3. Changing the models

The parameters of the field models are keys of the exp/run. Here is a variant of the standard scenario with a more contagious TYLCV, seeded from a single infection:

In [ ]:
(scenarios / "miniberry-30-contagious.yaml").write_text("""typename: "Miniberry-30"
precompute-time: 10
tylcv-p-transmission: 0.6
tylcv-infection-seeds: 1
""")
show_exprun("miniberry-30-contagious")
farm, wbfe = create_wbfe(precompute_environment("miniberry-30-contagious"))
show_fields(wbfe, [1, 4, 7, 10], fields=["tylcv"], title="miniberry-30-contagious")

**The cache knows its configuration.** A second `precompute_environment` with the same configuration replays, and `time_done` in its `exprun.yaml` is unchanged. After the exp/run is changed, the environment is recomputed.

In [ ]:
def time_done(run):
    exp_env = Config().get_experiment("environment", run)
    return yaml.safe_load(open(pathlib.Path(exp_env["data_dir"], "exprun.yaml")))["time_done"]

before = time_done("miniberry-30-contagious")
precompute_environment("miniberry-30-contagious")
print("same configuration, replayed:", time_done("miniberry-30-contagious") == before)
text = (scenarios / "miniberry-30-contagious.yaml").read_text()
(scenarios / "miniberry-30-contagious.yaml").write_text(text.replace("0.6", "0.3"))
precompute_environment("miniberry-30-contagious")
print("changed configuration, recomputed:", time_done("miniberry-30-contagious") != before)

## 4. Picture-based scenarios

A field can be given by a picture next to the exp/run (`<field>-picture`). The red or gray channel of the picture, divided by 255, gives the value: white (1.0) is healthy, mid-gray (0.5) infected, and black (0.0) destroyed. There are two modes:
- **`static`:** the picture is the field on every day.
- **`initial`** (TYLCV and CCR only): the picture is the initial state of the epidemic, which then evolves.

The two MRMR scenarios use static pictures on a tomato-only Miniberry-100 farm, with clustered or scattered patches of destroyed plants:

In [ ]:
show_exprun("mrmr-custom-clustered")
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, run in zip(axes, ["mrmr-custom-clustered", "mrmr-custom-notclustered"]):
    farm, wbfe = create_wbfe(precompute_environment(run))
    wbfe.proceed(25)
    ax.imshow(wbfe.tylcv.value.T, origin="lower", cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{run}, TYLCV day 25")
plt.show()

**Creating a picture.** A new exp/run can name a picture that does not exist yet. Creating the environment then writes the field of the model as a gray **template** with that name, and raises an exception naming the file.

In [ ]:
(scenarios / "my-scenario.yaml").write_text("""typename: "Miniberry-30"
planting: "tomato-only"
precompute-time: 10
tylcv-picture: "my-picture.png"
tylcv-picture-mode: "initial"
""")
try:
    precompute_environment("my-scenario")
except Exception as exception:
    print("Exception:", exception)

The template would be edited in an image editor such as GIMP. Here we paint it programmatically: two small infected (mid-gray) squares on a healthy field.

**Orientation:** a picture is indexed like the fields, as `[x, y]`. Its image row is x and its column is y, so in an image editor it appears transposed relative to the figures of this notebook, which have x to the right and y up.

In [ ]:
template = imageio.imread(scenarios / "my-picture.png")
print("template shape:", template.shape)
picture = np.ones((30, 30))
picture[5:8, 5:8] = 0.5     # x 5..7, y 5..7
picture[20:23, 18:21] = 0.5 # x 20..22, y 18..20
plt.imsave(scenarios / "my-picture.png", picture, cmap="gray", vmin=0, vmax=1)
farm, wbfe = create_wbfe(precompute_environment("my-scenario"))
show_fields(wbfe, [1, 3, 6, 10], fields=["tylcv"], title="my-scenario: initial mode, the squares grow into an epidemic")

The same picture in static mode stays as painted:

In [ ]:
text = (scenarios / "my-scenario.yaml").read_text()
(scenarios / "my-scenario.yaml").write_text(text.replace('"initial"', '"static"'))
farm, wbfe = create_wbfe(precompute_environment("my-scenario"))
show_fields(wbfe, [1, 10], fields=["tylcv"], title="my-scenario: static mode")

## 5. The cache

A cached environment stores every field of every precomputed day under the exp/run's data directory, in `precalc_<field>/env_value_<day>`, together with a JPG preview of each day:

In [ ]:
exp_env = Config().get_experiment("environment", "miniberry-30")
data_dir = pathlib.Path(exp_env["data_dir"])
for path in sorted(data_dir.iterdir()):
    print(path.name, f"({len(list(path.iterdir()))} files)" if path.is_dir() else "")

**The full Waterberry farm is not cached by default** (`cache: null`). Its cache would be too large, so its environment is computed live, from the same seeds, in every run.

In [ ]:
for run in ["miniberry-100", "benchmark-waterberry", "waterberry-tomato"]:
    print(f"{run:22s} cached: {uses_cache(Config().get_experiment('environment', run, create_data_dir=False))}")
gigabytes = 6000 * 5000 * 8 * 3 * 50 / 1e9
print(f"a full Waterberry cache: 6000 x 5000 float64 values x 3 fields x 50 days = about {gigabytes:.0f} GB")

**Replay is strict.** Replaying a day that was not precomputed raises an exception, instead of silently using stale or empty ground truth:

In [ ]:
farm, wbfe = create_wbfe(precompute_environment("miniberry-30"))
try:
    wbfe.proceed(exp_env["precompute-time"] + 1)
except Exception as exception:
    print("Exception:", exception)

## 6. Creating a scenario

| To change | Set in the environment exp/run |
|---|---|
| The size | `typename`: Miniberry-10 / 30 / 100 or Waterberry; a new size is a new `FarmGeometry` subclass |
| The crops | `planting`: standard, tomato-only or strawberry-only |
| The spread of a disease | `tylcv-*` / `ccr-*`: `p-transmission`, `infection-duration`, `infection-seeds`, `spread-dimension` (null: scaled with the grid) |
| The soil moisture | `soil-seed`, `soil-evaporation`, `soil-rainfall`, `soil-rain-likelihood` |
| A field from a picture | `<field>-picture` and, for the diseases, `<field>-picture-mode` (static or initial) |
| The randomness | `seed` (epidemics), `soil-seed` |
| How long it evolves | `precompute-time`, at least the experiments' `time-start-environment` |
| Caching | `cache`: true, false, or null (everything except the full Waterberry farm) |

Then call `precompute_environment(run)` and `create_wbfe(exp_env)`. Experiments refer to the scenario with `exp_environment: "environment"` and `run_environment: "<run>"`.

In [ ]:
shutil.rmtree(workspace)